# 01 — Supply Chain Network Graph
**Module A**

Builds a directed weighted supply chain graph, computes centrality metrics,
runs disruption simulations for 3 historical events, and creates interactive
Plotly network visualizations.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import networkx as nx
import plotly.graph_objects as go
from src.graph import (
    build_supply_chain_graph, compute_centrality_metrics,
    run_historical_simulations, create_network_plotly_figure
)
import duckdb

## Build the Supply Chain Graph

In [2]:
con = duckdb.connect('../data/processed/supply_chain.db', read_only=True)
G = build_supply_chain_graph(con)

  ✓ Built supply chain graph: 107 nodes, 186 edges


In [3]:
print(f'Nodes: {G.number_of_nodes()}')
print(f'Edges: {G.number_of_edges()}')
print(f'\nNode types:')
for ntype in ['supplier', 'manufacturer', 'distributor', 'retailer']:
    count = sum(1 for _, d in G.nodes(data=True) if d.get('node_type') == ntype)
    print(f'  {ntype}: {count}')

Nodes: 107
Edges: 186

Node types:
  supplier: 72
  manufacturer: 10
  distributor: 5
  retailer: 20


## Centrality Metrics

In [4]:
centrality = compute_centrality_metrics(G)
centrality.sort_values('betweenness_centrality', ascending=False).head(15)

  ✓ Computed centrality metrics for 107 nodes
    Critical nodes: 12
    High risk nodes: 95


,node_id,node_type,country,category,degree_centrality,betweenness_centrality,pagerank,clustering_coefficient,risk_tier
86,Latin_America,distributor,,,0.141509,0.036837,0.025742,0,Critical
84,EU_Central,distributor,,,0.141509,0.036837,0.084890,0,Critical
85,Asia_Pacific,distributor,,,0.141509,0.036837,0.042712,0,Critical
77,BASF_DE,manufacturer,Germany,,0.245283,0.024708,0.051338,0,Critical
78,Pfizer_US,manufacturer,USA,,0.150943,0.024708,0.036393,0,Critical
79,Nestlé_CH,manufacturer,Switzerland,,0.150943,0.024708,0.036393,0,Critical
80,ArcelorMittal_LU,manufacturer,Luxembourg,,0.141509,0.022462,0.018460,0,Critical
75,TSMC_TW,manufacturer,Taiwan,,0.132075,0.020216,0.012482,0,Critical
81,Inditex_ES,manufacturer,Spain,,0.132075,0.020216,0.030416,0,Critical
72,Toyota_JP,manufacturer,Japan,,0.150943,0.018778,0.019955,0,Critical


## Disruption Simulations

In [5]:
sim_results = run_historical_simulations(G)
sim_results[['event_name', 'nodes_removed', 'disconnected_pct', 
             'pct_trade_value_at_risk', 'new_chokepoints']]


── Running Historical Disruption Simulations ──

  ── Disruption Simulation: Ukraine Conflict (2022) ──
     Nodes removed: 3
     Disconnected nodes: 0 (0.0%)
     Trade value at risk: $1,062,600 (0.13%)

  ── Disruption Simulation: Red Sea Shipping Disruption (2023) ──
     Nodes removed: 0
     Disconnected nodes: 0 (0.0%)
     Trade value at risk: $0 (0.0%)

  ── Disruption Simulation: Russia Sanctions (2022) ──
     Nodes removed: 4
     Disconnected nodes: 0 (0.0%)
     Trade value at risk: $4,416,930 (0.52%)


,event_name,nodes_removed,disconnected_pct,pct_trade_value_at_risk,new_chokepoints
0,Ukraine Conflict (2022),3,0.0,0.13,"[Toyota_JP, TSMC_TW, BASF_DE, Pfizer_US, Nestl..."
1,Red Sea Shipping Disruption (2023),0,0.0,0.00,"[Toyota_JP, BASF_DE, Pfizer_US, Nestlé_CH, Arc..."
2,Russia Sanctions (2022),4,0.0,0.52,"[Toyota_JP, TSMC_TW, BASF_DE, Pfizer_US, Nestl..."


## Network Visualization

In [6]:
fig = create_network_plotly_figure(G, centrality)
fig.show()

In [7]:
con.close()